# Notebook 102 — Índice vectorial con Databricks AI Search

**Serie:** 100–106 — Agente RAG con evaluación en Databricks Free Edition

En el notebook 101 viste el techo de TF-IDF: falla cuando la pregunta y la evidencia dicen lo
mismo con palabras distintas. Aquí construyes el retriever que resuelve ese problema.

**Databricks AI Search** (antes Vector Search) es el servicio gestionado de búsqueda
vectorial de la plataforma. Lo que aporta sobre montar tu propia solución:

- Calcula los embeddings por ti, con un modelo servido en la misma plataforma
- Mantiene el índice sincronizado con la tabla Delta de origen
- Se gobierna con Unity Catalog, como cualquier otra tabla

## ⚠️ Ejecuta este notebook ANTES de la clase

La creación del índice tarda entre 5 y 15 minutos: hay que calcular el embedding de cada
chunk. No es tiempo de demostración en vivo. Los notebooks 103 en adelante solo consultan el
índice, y esos sí son inmediatos.

## 1. Dependencias

`databricks-vectorsearch` no viene preinstalado. Tras instalarlo hay que reiniciar el
intérprete de Python para que el import funcione.

In [0]:
#%pip install -q -U databricks-vectorsearch
%pip install -q -U databricks-ai-search
%restart_python

In [0]:
CATALOG = "big_data_ii_2025"
SCHEMA = "spark_examples"

T_CORPUS = f"{CATALOG}.{SCHEMA}.agenteval_corpus"
T_DEV = f"{CATALOG}.{SCHEMA}.agenteval_train_dev"

INDEX_NAME = f"{CATALOG}.{SCHEMA}.agenteval_corpus_index"
EMBEDDING_MODEL = "databricks-gte-large-en"
DEFAULT_ENDPOINT_NAME = "agenteval_endpoint"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

## 2. Verificar el prerequisito: Change Data Feed

Un índice **Delta Sync** se mantiene al día leyendo el Change Data Feed de la tabla de
origen. Así, cuando agregas o modificas chunks, solo se recalculan los embeddings de las
filas que cambiaron en vez de reconstruir el índice completo.

El notebook 100 ya habilitó el CDF. Esta celda lo confirma y lo corrige si hace falta, para
que el notebook funcione aunque lo ejecutes fuera de orden.

In [0]:
props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {T_CORPUS}").collect()}
cdf_enabled = props.get("delta.enableChangeDataFeed", "false").lower() == "true"

if not cdf_enabled:
    print("CDF no estaba habilitado. Habilitándolo...")
    spark.sql(f"ALTER TABLE {T_CORPUS} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Change Data Feed en {T_CORPUS}: habilitado")
print(f"Chunks a indexar: {spark.table(T_CORPUS).count():,}")

> **Antes de ejecutar:** El índice va a calcular un embedding por cada chunk del corpus. Cada
> embedding de `databricks-gte-large-en` es un vector de 1024 dimensiones.
>
> ¿Por qué crees que este servicio exige Change Data Feed en la tabla de origen, en lugar de
> simplemente releer la tabla completa cada vez que se sincroniza? Piensa en qué pasaría con
> un corpus de 10 millones de chunks al que le agregas 50.

## 3. Endpoint de AI Search

**Límite de Free Edition: un solo endpoint, con una sola unidad de búsqueda.**

Por eso la celda no crea un endpoint a ciegas: primero lista los existentes y reutiliza el
que haya. Si intentas crear un segundo, la operación falla con un error de cuota.

Si ya tienes un endpoint de otro ejercicio y quieres empezar limpio, elimina primero los
índices asociados desde **Catalog Explorer**.

In [0]:
from databricks.ai_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

existing = vsc.list_endpoints().get("endpoints", [])
existing_names = [e["name"] for e in existing]

if existing_names:
    VS_ENDPOINT = existing_names[0]
    print(f"Reutilizando endpoint existente: {VS_ENDPOINT}")
    if len(existing_names) > 1:
        print(f"  (hay más de uno: {existing_names})")
else:
    VS_ENDPOINT = DEFAULT_ENDPOINT_NAME
    print(f"No hay endpoints. Creando '{VS_ENDPOINT}' (tarda unos minutos)...")
    vsc.create_endpoint_and_wait(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print("Endpoint creado.")

print(f"\nEndpoint en uso: {VS_ENDPOINT}")

## 4. Crear el índice Delta Sync

Dos parámetros merecen atención:

**`embedding_source_column="chunk_text"`** — le entregas texto plano y Databricks se encarga
de vectorizarlo, tanto al indexar como al consultar. La alternativa (Direct Vector Access,
donde tú calculas y subes los vectores) **no está disponible en Free Edition**, así que este
es el camino obligado. También es el más conveniente: no tienes que preocuparte por usar el
mismo modelo de embeddings en indexación y consulta.

**`pipeline_type="TRIGGERED"`** — el índice se sincroniza cuando tú lo pides. La alternativa,
`CONTINUOUS`, mantiene un cluster de streaming encendido permanentemente: innecesario para un
corpus estático y costoso en cualquier edición.

In [0]:
def index_exists(client, endpoint: str, name: str) -> bool:
    try:
        client.get_index(endpoint_name=endpoint, index_name=name).describe()
        return True
    except Exception:
        return False


if index_exists(vsc, VS_ENDPOINT, INDEX_NAME):
    print(f"El índice {INDEX_NAME} ya existe. No se recrea.")
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
else:
    print(f"Creando índice {INDEX_NAME}...")
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=T_CORPUS,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
    )
    print("Índice creado. Ahora hay que esperar a que se pueble.")

## 5. Esperar a que el índice esté listo

Aquí es donde se van los minutos: el servicio está calculando un embedding por chunk. La
celda hace polling e informa el progreso.

Si el timeout se agota, no significa que algo falló necesariamente — puedes volver a ejecutar
esta celda, o revisar el estado en la UI (**Catalog Explorer → agenteval_corpus_index**).

In [0]:
import time

TIMEOUT_MIN = 35
deadline = time.time() + TIMEOUT_MIN * 60
last_state = None

while time.time() < deadline:
    desc = index.describe()
    status = desc.get("status", {})
    state = status.get("detailed_state", status.get("state", "UNKNOWN"))
    ready = status.get("ready", False)
    indexed = status.get("indexed_row_count", 0)

    if state != last_state:
        print(f"[{time.strftime('%H:%M:%S')}] estado={state}  filas_indexadas={indexed:,}")
        last_state = state

    if ready and "ONLINE" in str(state).upper():
        print(f"\nÍndice listo. {indexed:,} chunks indexados.")
        break
    if "FAILED" in str(state).upper():
        raise RuntimeError(f"La creación del índice falló: {status}")

    time.sleep(20)
else:
    print(
        f"\nSe agotó la espera de {TIMEOUT_MIN} min. El índice puede seguir poblándose.\n"
        "Vuelve a ejecutar esta celda, o revisa el estado en Catalog Explorer."
    )

## 6. Primera consulta semántica

In [0]:
SEARCH_COLUMNS = ["chunk_id", "doc_id", "title", "doc_type", "chunk_text"]


def search_vector(query: str, k: int = 3, filters: dict | None = None) -> list[dict]:
    """Consulta el índice y devuelve una lista de dicts, uno por chunk."""
    res = index.similarity_search(
        query_text=query, columns=SEARCH_COLUMNS, num_results=k, filters=filters or {}
    )
    rows = res.get("result", {}).get("data_array", [])
    cols = SEARCH_COLUMNS + ["score"]
    return [dict(zip(cols, r)) for r in rows]


demo_q = spark.table(T_DEV).select("question").first()["question"]
print(f"Pregunta: {demo_q}\n")
print("Resultados de AI Search:")
for c in search_vector(demo_q, k=3):
    print(f"  [{c['score']:.4f}] {c['chunk_id']}  ({c['doc_type']})")
    print(f"          {c['chunk_text'][:130]}...")

## 7. Comparación directa: TF-IDF vs vectorial

Reconstruimos el retriever TF-IDF desde el modelo guardado en el notebook 101 y comparamos
ambos sobre la misma muestra de preguntas. Todo el cálculo es local: **cero llamadas al LLM**.

In [0]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from pyspark.ml import PipelineModel

VOL = f"/Volumes/{CATALOG}/{SCHEMA}/agenteval"
N_FEATURES = 1 << 14

tfidf_model = PipelineModel.load(f"{VOL}/models/tfidf")
corpus_vec = tfidf_model.transform(spark.table(T_CORPUS)).select("chunk_id", "tfidf").collect()

CHUNK_IDS = [r["chunk_id"] for r in corpus_vec]
indptr, indices, data = [0], [], []
for r in corpus_vec:
    v = r["tfidf"]
    indices.extend(v.indices.tolist())
    data.extend(v.values.tolist())
    indptr.append(len(indices))
CORPUS_MATRIX = normalize(
    csr_matrix((data, indices, indptr), shape=(len(corpus_vec), N_FEATURES)), norm="l2", axis=1
)


def search_tfidf(question: str, k: int = 3) -> list[str]:
    q_vec = (tfidf_model.transform(spark.createDataFrame([(question,)], ["chunk_text"]))
             .select("tfidf").collect()[0]["tfidf"])
    q = normalize(
        csr_matrix((q_vec.values.tolist(), q_vec.indices.tolist(), [0, len(q_vec.indices)]),
                   shape=(1, N_FEATURES)),
        norm="l2",
    )
    scores = (CORPUS_MATRIX @ q.T).toarray().ravel()
    return [CHUNK_IDS[i] for i in np.argsort(-scores)[:k]]

print("Retriever TF-IDF reconstruido desde el modelo guardado.")

In [0]:
import pandas as pd

K = 3
sample = spark.table(T_DEV).orderBy("question_id").limit(25).collect()

rows = []
for r in sample:
    gold = set(r["gold_chunk_ids"])
    tf_ids = search_tfidf(r["question"], K)
    vec_ids = [c["chunk_id"] for c in search_vector(r["question"], K)]
    rows.append({
        "question_id": r["question_id"],
        "tfidf_hit": 1.0 if gold & set(tf_ids) else 0.0,
        "vector_hit": 1.0 if gold & set(vec_ids) else 0.0,
        "coincidencia": len(set(tf_ids) & set(vec_ids)) / K,
    })

comp = pd.DataFrame(rows)

print("=" * 52)
print(f"  RETRIEVAL: TF-IDF vs AI SEARCH  (k={K}, n={len(comp)})")
print("=" * 52)
print(f"  hit_rate TF-IDF      : {comp['tfidf_hit'].mean():.3f}")
print(f"  hit_rate AI Search   : {comp['vector_hit'].mean():.3f}")
print(f"  solapamiento de tops : {comp['coincidencia'].mean():.3f}")
print("=" * 52)

n_rescatadas = len(comp[(comp["tfidf_hit"] == 0) & (comp["vector_hit"] == 1)])
n_perdidas = len(comp[(comp["tfidf_hit"] == 1) & (comp["vector_hit"] == 0)])
print(f"\n  Rescatadas por el vectorial (TF-IDF fallaba): {n_rescatadas}")
print(f"  Perdidas por el vectorial (TF-IDF acertaba) : {n_perdidas}")

El solapamiento parcial es la observación interesante: los dos retrievers **no** son
intercambiables, encuentran cosas distintas. Que existan preguntas rescatadas por el
vectorial y otras perdidas sugiere que un sistema de producción podría combinarlos
(búsqueda híbrida) en vez de elegir uno.

Nota metodológica: 25 preguntas es una muestra pequeña. La diferencia observada es
orientativa, no concluyente — no reportes esto como evidencia definitiva de superioridad.

## 8. Filtros de metadatos

El índice hereda las columnas de la tabla, así que puedes restringir la búsqueda por
metadatos. Filtrar por `doc_type` reduce el espacio de búsqueda y elimina falsos positivos
de categorías irrelevantes.

In [0]:
doc_types = [r["doc_type"] for r in
             spark.table(T_CORPUS).select("doc_type").distinct().orderBy("doc_type").collect()]
print(f"doc_types disponibles: {doc_types}\n")

target = doc_types[0]
print(f"Búsqueda SIN filtro:")
for c in search_vector(demo_q, k=3):
    print(f"  {c['chunk_id']:<28} ({c['doc_type']})")

print(f"\nBúsqueda CON filtro doc_type='{target}':")
for c in search_vector(demo_q, k=3, filters={"doc_type": target}):
    print(f"  {c['chunk_id']:<28} ({c['doc_type']})")

## 9. Estado final y verificación en la UI

Puedes ver el índice en **Catalog → big_data_ii_2025 → spark_examples → agenteval_corpus_index**.
La pestaña **Overview** muestra el estado de sincronización, el modelo de embeddings y el
número de filas indexadas.

In [0]:
desc = index.describe()
status = desc.get("status", {})

print(f"Índice     : {INDEX_NAME}")
print(f"Endpoint   : {VS_ENDPOINT}")
print(f"Tipo       : {desc.get('index_type')}")
print(f"Estado     : {status.get('detailed_state', status.get('state'))}")
print(f"Filas      : {status.get('indexed_row_count', 0):,}")
print(f"Embeddings : {EMBEDDING_MODEL}")